In [2]:
import subprocess
import sys

def run_cmd(cmd):
    print(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

# 1. Clone the repo to the expected location (ignoring error if it already exists)
run_cmd("git clone https://github.com/overlordxrz-source/throng.git /root/throng || true")

# 2. Checkout our Phase 18 branch
run_cmd("cd /root/throng && git fetch && git checkout feature/phase18-crafting")

# 3. Install the required Python packages
run_cmd("pip install -r /root/throng/requirements.txt")

print("✅ Setup complete! You are ready to launch.")

Running: git clone https://github.com/overlordxrz-source/throng.git /root/throng || true
STDERR: Cloning into '/root/throng'...

Running: cd /root/throng && git fetch && git checkout feature/phase18-crafting
branch 'feature/phase18-crafting' set up to track 'origin/feature/phase18-crafting'.

STDERR: Switched to a new branch 'feature/phase18-crafting'

Running: pip install -r /root/throng/requirements.txt
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/26.8 MB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 17.8/26.8 MB 116.4 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 96.7 MB/s eta 0:00:00

STDERR: 
[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

✅ Setup complete! You are ready to launch.


In [3]:
import subprocess
import time

print("Pulling latest code...")
result = subprocess.run("cd /root/throng && git pull origin feature/phase18-crafting", shell=True, capture_output=True, text=True)
print(result.stdout)

# 1. Safely kill previous runs
print("Killing old training processes...")
subprocess.run("pkill -f 'python -u.*run_bg.py'", shell=True)
subprocess.run("pkill -f 'python -u.*/root/throng/scripts/modal_train.py'", shell=True)
time.sleep(2)

# 2. VERIFICATION: Ensure the process count is 0 before launching (HARD RULE)
pgrep_res = subprocess.run("pgrep -af 'python -u'", shell=True, capture_output=True, text=True)
if pgrep_res.stdout.strip():
    print("⚠️ WARNING: Stray Python processes still running:")
    print(pgrep_res.stdout)
else:
    print("✅ Process wipe verified. Clean slate.")

# 3. Relaunch with Cam's required environment variables
launch_cmd = """
cd /root/throng && \
export TF_GPU_ALLOCATOR=cuda_malloc_async && \
export XLA_PYTHON_CLIENT_MEM_FRACTION=0.80 && \
export JAX_COMPILATION_CACHE_DIR=/tmp/throng_jax_cache && \
nohup python -u run_bg.py > /mnt/throng-runs/train.log 2>&1 &
"""

p = subprocess.Popen(
    launch_cmd,
    shell=True,
    start_new_session=True  # MANDATORY
)
print("✅ Phase 18 Launched securely in a new session!")

Pulling latest code...
Already up to date.

Killing old training processes...
⚠️ WARNING: Stray Python processes still running:
258 /bin/sh -c pgrep -af 'python -u'

✅ Phase 18 Launched securely in a new session!


In [4]:
run_cmd("cd /root/throng && git fetch --all")
run_cmd("cd /root/throng && git checkout feature/phase18-crafting")
run_cmd("cd /root/throng && git pull origin feature/phase18-crafting")

Running: cd /root/throng && git fetch --all
Running: cd /root/throng && git checkout feature/phase18-crafting
Your branch is up to date with 'origin/feature/phase18-crafting'.

STDERR: Already on 'feature/phase18-crafting'

Running: cd /root/throng && git pull origin feature/phase18-crafting
Already up to date.

STDERR: From https://github.com/overlordxrz-source/throng
 * branch            feature/phase18-crafting -> FETCH_HEAD



In [5]:
import subprocess

# Force sync to the remote tip, ignoring local history
sync_cmd = "cd /root/throng && git fetch origin && git reset --hard origin/feature/phase18-crafting"
print(subprocess.run(sync_cmd, shell=True, capture_output=True, text=True).stdout)

# Kill the broken run safely
subprocess.run("pkill -f 'python -u.*run_bg.py'", shell=True)
subprocess.run("pkill -f 'python -u.*/root/throng/scripts/modal_train.py'", shell=True)

# Relaunch
launch_cmd = """
cd /root/throng && \
export TF_GPU_ALLOCATOR=cuda_malloc_async && \
export XLA_PYTHON_CLIENT_MEM_FRACTION=0.80 && \
export JAX_COMPILATION_CACHE_DIR=/tmp/throng_jax_cache && \
nohup python -u run_bg.py > /mnt/throng-runs/train.log 2>&1 &
"""

subprocess.Popen(launch_cmd, shell=True, start_new_session=True)
print("✅ Hard reset complete. Phase 18 relaunched on commit dffe4df!")

HEAD is now at c4325f6 Fix Phase 18 own_state_dim bug in causal_intervention.py

✅ Hard reset complete. Phase 18 relaunched on commit dffe4df!


In [6]:
import subprocess, os, time

# 1. Patch is already in the repo — pull it
result = subprocess.run(
    ["git", "-C", "/root/throng", "pull", "origin", "feature/phase18-crafting"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

# 2. Confirm the bincount fix is present
check = subprocess.run(
    ["grep", "-n", "toks_s0", "/root/throng/jax_sim/main_jax.py"],
    capture_output=True, text=True
)
print("Patch confirmed:" if check.stdout else "❌ PATCH NOT FOUND — do not proceed")
print(check.stdout)

# 3. Kill any zombies
subprocess.run(["pkill", "-f", "run_bg.py"], capture_output=True)
time.sleep(2)
remaining = subprocess.run(["pgrep", "-f", "run_bg.py"], capture_output=True, text=True)
print(f"Remaining run_bg processes: '{remaining.stdout.strip()}' (should be empty)")

# 4. Relaunch
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.80"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/throng_jax_cache"

proc = subprocess.Popen(
    ["python", "-u", "/root/throng/run_bg.py"],
    stdout=open("/mnt/throng-runs/train.log", "a"),
    stderr=subprocess.STDOUT,
)
print(f"✅ Launched run_bg.py (pid {proc.pid})")


Already up to date.

From https://github.com/overlordxrz-source/throng
 * branch            feature/phase18-crafting -> FETCH_HEAD

Patch confirmed:
2317:                            toks_s0 = np.asarray(toks)
2318:                            if toks_s0.ndim > 1:
2320:                                for slot_idx in range(toks_s0.shape[1]):
2321:                                    majority_toks.append(int(np.bincount(toks_s0[:, slot_idx].astype(int)).argmax()))
2324:                                nb_scout_token_lag1[i] = int(np.bincount(toks_s0.astype(int)).argmax())
2422:                            toks_s0 = np.asarray(toks)
2423:                            if toks_s0.ndim > 1:
2424:                                toks_s0 = toks_s0[:, 0]
2425:                            nb_hunter_token_lag1[i] = int(np.bincount(toks_s0.astype(int)).argmax())

Remaining run_bg processes: '' (should be empty)
✅ Launched run_bg.py (pid 311)


In [8]:
!tail -f -n 60 /mnt/throng-runs/train.log

run_bg.py — THRONG | blue: 3-slot discrete VQ (12/8/12, 64-code) on a 32-D effective wire, 12 actions, GWT-masked comms | red: pure ecological pressure (VQ decoupled) | hot-resume from checkpoint | n_steps=3_000_000
[JAX] Phase14.2 Metabolic Asymmetry: red_energy_decay=0.0001
[JAX] Corpus persistence: /mnt/throng-runs/signal_corpus.jsonl
[JAX] Corpus persistence: /mnt/throng-runs/signal_corpus_red.jsonl
[JAX] Red corpus: signal_corpus_red.jsonl (hunter = blue_dist <= hunt_scout_range=8.0)
[JAX] obs_dim = 2647
[JAX] code: /root/throng/jax_sim/main_jax.py
[JAX] git=c4325f6 | Phase9 auxiliary: ON (AuxLoss line on dashboard)
[JAX] Phase11 carry_fwd: head_fwd_dyn_1/2 → carry_{t+1} MSE (stop_grad target)
[JAX] Phase12.1 spatial epistemic gate: conf_pred < mean(conf|alive)*1.0 → imagined_action (batch-relative, stateless); else reactive | K=5 γ=0.999
[JAX] red_sense_api=v2 (observations_jax)
[JAX] Phase12 red comms: PredatorNetworkJax hidden=128 red_codebook vocab=64 cross_attn=True heads=4 |